# Gambling Domain Classifier — Colab Training

Alternative pipeline using `tflite-model-maker` (AverageWordVec).
For the main pipeline with Conv1D + numeric features, use `train.py`.

**Requirements:**
- Google Colab (free, no GPU needed)
- Dataset: `dataset_balanced.csv` (upload in cell 2)

**Output:** `gambling_classifier.tflite` (~1-3 MB)

## 1. Install Python 3.9 via Conda

`tflite-model-maker` does not support Python 3.10+. We install Python 3.9 via Miniconda.

In [ ]:
# Download and install Miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /opt/conda 2>/dev/null || true

# Accept terms of service
!/opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
!/opt/conda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Create Python 3.9 environment
!/opt/conda/bin/conda create -n py39 python=3.9 -y -q

print("\n=== Python 3.9 installed ===")
!/opt/conda/envs/py39/bin/python --version

## 2. Install dependencies

In [ ]:
!/opt/conda/envs/py39/bin/pip install -q tflite-model-maker matplotlib-inline numpy\<1.24
print("\n=== Dependencies installed ===")

## 3. Upload dataset

Upload the `dataset_balanced.csv` file.

In [ ]:
from google.colab import files
import os

if not os.path.exists("dataset_balanced.csv"):
    uploaded = files.upload()
    print(f"\nFile received: {list(uploaded.keys())}")
else:
    print("Dataset already present.")

!wc -l dataset_balanced.csv

## 4. Train the model

Creates the training script and runs it with Python 3.9.

In [ ]:
%%writefile train_colab.py
import os
import sys
import time

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["MPLBACKEND"] = "Agg"

print("="*60)
print("  Gambling Detector - TFLite Training (AverageWordVec)")
print("="*60)

DATASET = "dataset_balanced.csv"
EPOCHS = 20
BATCH_SIZE = 32
SPLIT = 0.9
OUTPUT_DIR = "output"
MODEL_NAME = "gambling_classifier.tflite"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n[1/4] Importing TFLite Model Maker...")
t0 = time.time()
from tflite_model_maker import text_classifier
from tflite_model_maker.text_classifier import AverageWordVecSpec
print(f"       OK ({time.time()-t0:.1f}s)")

print("\n[2/4] Loading dataset...")
t0 = time.time()
spec = AverageWordVecSpec()
data = text_classifier.DataLoader.from_csv(
    DATASET,
    text_column="text_column",
    label_column="label",
    model_spec=spec,
    delimiter=",",
)
train_data, test_data = data.split(SPLIT)
print(f"       Train: {len(train_data)} samples")
print(f"       Test:  {len(test_data)} samples")
print(f"       OK ({time.time()-t0:.1f}s)")

print(f"\n[3/4] Training model ({EPOCHS} epochs)...")
t0 = time.time()
model = text_classifier.create(
    train_data,
    model_spec=spec,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)
train_time = time.time() - t0
print(f"       Training completed in {train_time:.1f}s")

print("\n[4/4] Evaluating and exporting...")
loss, accuracy = model.evaluate(test_data)
print(f"       Loss:     {loss:.4f}")
print(f"       Accuracy: {accuracy:.2%}")

model.export(export_dir=OUTPUT_DIR, tflite_filename=MODEL_NAME)

tflite_path = os.path.join(OUTPUT_DIR, MODEL_NAME)
size_mb = os.path.getsize(tflite_path) / (1024*1024)

print("\n" + "="*60)
print("  RESULT")
print("="*60)
print(f"  Accuracy:    {accuracy:.2%}  {'OK' if accuracy >= 0.90 else 'BELOW target (90%)'}")
print(f"  Size:        {size_mb:.2f} MB  {'OK' if size_mb < 5 else 'ABOVE limit (5 MB)'}")
print(f"  Train time:  {train_time:.1f}s")
print(f"  Output:      {tflite_path}")
print("="*60)

In [ ]:
# Run training with Python 3.9
!/opt/conda/envs/py39/bin/python train_colab.py

## 5. Verify generated model

In [ ]:
import os

model_path = "output/gambling_classifier.tflite"

if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / (1024*1024)
    print(f"Model found: {model_path}")
    print(f"Size: {size_mb:.2f} MB")
    print(f"Status: {'OK - ready to use!' if size_mb < 5 else 'WARNING: above 5 MB'}")
else:
    print("ERROR: Model was not generated. Check the logs above.")

## 6. Download model

In [ ]:
from google.colab import files

model_path = "output/gambling_classifier.tflite"
if os.path.exists(model_path):
    files.download(model_path)
    print("Download started!")
    print("\nGenerated files:")
    print("  - gambling_classifier.tflite (model)")
    print("  - output/vocab.txt (vocabulary)")
    print("  - output/labels.txt (labels)")
else:
    print("Model not found. Run the training cell first.")